In [ ]:
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import re

In [ ]:
def build_initial_data(directory, common, regex):
    # read image metadata, mainly file location
    images = pd.read_csv(directory / 'Image.csv', usecols=['Metadata_FileLocation', 'ImageNumber', 'Metadata_Series'])
    
    # add in regex of filename (time, treatment, etc)
    if regex:
        images = images.join(images['Metadata_FileLocation'].str.extract(regex))
    
    return images

def add_rim(result, directory, common, num_bins=1, total=20):
    cols = open(directory / 'InitialNucleolus.csv').readline().split(',')
    cols = [c for c in cols 
            if c.startswith('RadialDistribution_FracAtD')
           ]
    distributions = pd.read_csv(
        directory / 'InitialNucleolus.csv', 
        usecols=common + cols,
    )
    
    bins = [i for i in range(total, total-num_bins, -1)]
    relative_areas = distributions[[f'RadialDistribution_FracAtD_NucleolusObjectImage_{bin}of{total}' for bin in bins]].sum(axis=1)
    distributions['RPL_rim_enrichment'] = distributions[[f'RadialDistribution_FracAtD_RPL_{bin}of{total}' for bin in bins]].sum(axis=1) / relative_areas
    bins = [i for i in range(1, num_bins+1)]
    relative_areas = distributions[[f'RadialDistribution_FracAtD_NucleolusObjectImage_{bin}of{total}' for bin in bins]].sum(axis=1)
    distributions['RPL_core_enrichment'] = distributions[[f'RadialDistribution_FracAtD_RPL_{bin}of{total}' for bin in bins]].sum(axis=1) / relative_areas

    return result.merge(distributions.filter(regex='^(?!Radial)'), on='ImageNumber')
    
def read_data(directory, regex=None, bins=4):
    directory = Path(directory)
    # image and object number are uniuqe identifiers.  Area is used a lot and the parent_mergedGC should corresopnd to a single cell
    common = ['ImageNumber', 'ObjectNumber',]

    result = build_initial_data(directory, common, regex)
    
    result = add_rim(result, directory, common, bins, total=20)
    
    return result

full_data = read_data('outputs', r'/[A-G]\d+_L10A_0p(?P<time>\d)c.*nd2', bins=3)
full_data.columns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
sns.lineplot(data=full_data, x='time', y='RPL_core_enrichment', ax=axes[0])
sns.lineplot(data=full_data, x='time', y='RPL_rim_enrichment', ax=axes[1])